# Lab 2
## Jonathan Brooks - 14.02.2022

### Ex1

#### a)

In [130]:
import nltk
from nltk.corpus import brown

def getBrownWordsWithTag():
    return brown.tagged_words(tagset="universal")

def getMostCommonTagsInBrown(brownWords, nmbr):
    tags = [w[1] for w in brownWords]
    return nltk.FreqDist(tags).most_common(nmbr)

print(getMostCommonTagsInBrown(getBrownWordsWithTag(), 5))


[('NOUN', 275558), ('VERB', 182750), ('.', 147565), ('ADP', 144766), ('DET', 137019)]


Output

`[('NOUN', 275558), ('VERB', 182750), ('.', 147565), ('ADP', 144766), ('DET', 137019)]`

#### b)

In [131]:
def getFreqDistForBrown():
    return nltk.ConditionalFreqDist((w.lower(), tag) for (w, tag) in getBrownWordsWithTag())

def getAmbigousWordsCount(data):
    count = 0
    for w in data.conditions():
        if len(data[w].most_common()) > 2:
            count += 1
    
    return count

print(getAmbigousWordsCount(getFreqDistForBrown()))

274


Output

`274` 

#### c)

In [132]:
def getAmbigousWordPercentage():
    data = getFreqDistForBrown()
    return getAmbigousWordsCount(data)/len(data)

print(getAmbigousWordPercentage())

0.005500351299809294


Output

`0.005500351299809294` 

#### d)

In [133]:
def topFiveWordsWithMostDistinctTags():
    freq = getFreqDistForBrown()
    sortedFreq = sorted(freq, reverse=True, key=lambda w: len(freq[w]))
    words = []
    x = 0
    while len(words) < 5:
        if len(sortedFreq[x]) > 4:
            words.append(sortedFreq[x])
        x += 1

    return words


def getBrownTaggedSentences():
    return brown.tagged_sents(tagset="universal")

def getSentencesWithDifferentUsesOfWord(word):
    tags = getFreqDistForBrown()[word].most_common()
    taggedSentences = getBrownTaggedSentences()
    print("WORD IS: " + word)
    print("SENTENCE FOR EACH TAG:")
    for tag in tags:
        print((word, tag[0]))
        for sent in taggedSentences:
            wordsInLower = [(w[0].lower(), w[1]) for w in sent]
            if (word, tag[0]) in wordsInLower:
                print(" ".join([w[0] for w in sent]))
                break


topFive = topFiveWordsWithMostDistinctTags()
getSentencesWithDifferentUsesOfWord(topFive[0])


WORD IS: round
SENTENCE FOR EACH TAG:
('round', 'ADJ')
`` It was a nice round figure , that 12 '' , he said as he headed for the clubhouse , not too much perturbed .
('round', 'NOUN')
W. Averell Harriman of the United States , Malcolm MacDonald of Britain , Maurice Couve De Murville , France's Foreign Minister , and Howard C. Green , Canada's Minister of External Affairs , concluded , meanwhile , a round of consultations here on future tactics in the conference .
('round', 'ADV')
The gardens themselves are open free of charge the year round , and the 192 permanent employes make sure that not a dead or wilted flower is ever seen indoors or out by any visitor .
('round', 'VERB')
Bill Jr. , 20 , Kathy , 15 , and Patricia , 9 , round out the Shea family .
('round', 'ADP')
He is a parent with a child to nurture , here and now , and he is an educator who worries about the children half way round the world .


Output

```
WORD IS: round
SENTENCE FOR EACH TAG:
('round', 'ADJ')
`` It was a nice round figure , that 12 '' , he said as he headed for the clubhouse , not too much perturbed .
('round', 'NOUN')
W. Averell Harriman of the United States , Malcolm MacDonald of Britain , Maurice Couve De Murville , France's Foreign Minister , and Howard C. Green , Canada's Minister of External Affairs , concluded , meanwhile , a round of consultations here on future tactics in the conference .
('round', 'ADV')
The gardens themselves are open free of charge the year round , and the 192 permanent employes make sure that not a dead or wilted flower is ever seen indoors or out by any visitor .
('round', 'VERB')
Bill Jr. , 20 , Kathy , 15 , and Patricia , 9 , round out the Shea family .
('round', 'ADP')
He is a parent with a child to nurture , here and now , and he is an educator who worries about the children half way round the world .
````

#### e)

To solve the problem of ambiguous words on a predictive smart-keyboard I would probably first start of by analysing what type of sentence the user has typed up to that point, then compare that data to other sentences to predict what word could be next. 

### Ex 2

#### a)

In [134]:
from nltk.corpus import nps_chat
from sklearn.model_selection import train_test_split as split
from nltk.tag import DefaultTagger

def getNPSChatPostsWithTag():
    return nps_chat.tagged_posts(tagset="universal")

def getTrainTestDataBrown(ratio):
    return split(getBrownTaggedSentences(), test_size=ratio)

def getTrainTestDataNPS(ratio):
    return split(getNPSChatPostsWithTag(), test_size=ratio)

def getMostCommonTag(words):
    tags = [tag for e in words for (word, tag) in e]
    return nltk.FreqDist(tags).max()

datasetsTrain = {}
datasetsTest = {}

for r in [0.9, 0.5]:
    brownTrain, brownTest = getTrainTestDataBrown(r)
    npsTrain, npsTest = getTrainTestDataNPS(1-r)
    datasetTrain = brownTrain + npsTrain
    datasetTest = brownTest + npsTest
    datasetsTrain["Brown "+ str(int(r*100)) +"% | NPS " + str(int(100 - (r*100))) + "%"] = datasetTrain
    datasetsTest["Brown "+ str(int(r*100)) +"% | NPS " + str(int(100 - (r*100))) + "%"] = datasetTest

for r in [0.9, 0.5]:
    npsTrain, npsTest = getTrainTestDataNPS(r)
    brownTrain, brownTest = getTrainTestDataBrown(1-r)
    datasetTrain = npsTrain + brownTrain
    datasetTest = npsTest + brownTest
    datasetsTest["NPS "+ str(int(r*100)) +"% | BROWN " + str(int(100 - (r*100))) + "%"] = datasetTest
    datasetsTrain["NPS "+ str(int(r*100)) +"% | BROWN " + str(int(100 - (r*100))) + "%"] = datasetTrain

def defaultTagger(tag):
    tagger = DefaultTagger(tag)
    return tagger

def testerA():
    brownMostCommonTag = getMostCommonTag(getBrownTaggedSentences())
    npsMostCommonTag = getMostCommonTag(getNPSChatPostsWithTag())

    for t in [brownMostCommonTag, npsMostCommonTag]:
        print("TAG: " + t)

        for dKey in datasetsTest.keys():
            print(dKey)
            result = defaultTagger(t)
            result.tag(datasetsTest[dKey])
            print("Accuracy: " + str(result.accuracy(datasetsTest[dKey])))
        
        print("-----------")

testerA()


TAG: NOUN
Brown 90% | NPS 10%
Accuracy: 0.23718751872961147
Brown 50% | NPS 50%
Accuracy: 0.23692370312520683
NPS 90% | BROWN 10%
Accuracy: 0.23142455422764746
NPS 50% | BROWN 50%
Accuracy: 0.2367564442335264
-----------
TAG: NOUN
Brown 90% | NPS 10%
Accuracy: 0.23718751872961147
Brown 50% | NPS 50%
Accuracy: 0.23692370312520683
NPS 90% | BROWN 10%
Accuracy: 0.23142455422764746
NPS 50% | BROWN 50%
Accuracy: 0.2367564442335264
-----------


Output

````
TAG: NOUN
Brown 90% | NPS 10%
Accuracy: 0.23718751872961147
Brown 50% | NPS 50%
Accuracy: 0.23692370312520683
NPS 90% | BROWN 10%
Accuracy: 0.23142455422764746
NPS 50% | BROWN 50%
Accuracy: 0.2367564442335264
-----------
TAG: NOUN
Brown 90% | NPS 10%
Accuracy: 0.23718751872961147
Brown 50% | NPS 50%
Accuracy: 0.23692370312520683
NPS 90% | BROWN 10%
Accuracy: 0.23142455422764746
NPS 50% | BROWN 50%
Accuracy: 0.2367564442335264
-----------
````


#### b)

In [135]:
from nltk.tag import RegexpTagger, UnigramTagger, BigramTagger

patterns = [
(r'.*ing$', 'VBG'),                # gerunds
(r'.*ed$', 'VBD'),                 # simple past
(r'.*es$', 'VBZ'),                 # 3rd singular present
(r'.*ould$', 'MD'),                # modals
(r'.*\'s$', 'NN$'),                # possessive nouns
(r'.*s$', 'NNS'),                  # plural nouns
(r'^-?[0-9]+(\.[0-9]+)?$', 'CD'),  # cardinal numbers
(r'.*', 'NN')                      # nouns (default)
]
    

def regexTaggerWithBackoff(backoff):
    tagger = RegexpTagger(patterns, backoff=backoff)
    return tagger

def runRegexWithDefault():
    print("Regex tagger, with default tagger as backoff")
    for dKey in datasetsTest.keys():
        print(dKey)
        result = regexTaggerWithBackoff(defaultTagger(getMostCommonTag(datasetsTrain[dKey])))
        print("Accuracy: " + str(result.accuracy(datasetsTest[dKey])))
    

def unigramTaggerWithBigramBackoffWithRegexBackoff(train):
    reg = RegexpTagger(patterns)
    bi = BigramTagger(train, backoff=reg)
    uni = UnigramTagger(train, backoff=bi)
    return uni

def runRegexWithNGramBackoffs():
    print("N-gram taggers with regex taggers as backoff")
    for dKey in datasetsTest.keys():
        print(dKey)
        # Decided to put the regextagger as a backoff to the bigram tagger, as when I tried having the regex as the main tagger I got the same accuracy as above.
        result = unigramTaggerWithBigramBackoffWithRegexBackoff(datasetsTrain[dKey])
        print("Accuracy: " + str(result.accuracy(datasetsTest[dKey])))
    

runRegexWithDefault()
runRegexWithNGramBackoffs()



Regex tagger, with default tagger as backoff
Brown 90% | NPS 10%
Accuracy: 0.0
Brown 50% | NPS 50%
Accuracy: 0.0
NPS 90% | BROWN 10%
Accuracy: 0.0
NPS 50% | BROWN 50%
Accuracy: 0.0
N-gram taggers with regex taggers as backoff
Brown 90% | NPS 10%
Accuracy: 0.7063558431155419
Brown 50% | NPS 50%
Accuracy: 0.8230555151097992
NPS 90% | BROWN 10%
Accuracy: 0.799443982872116
NPS 50% | BROWN 50%
Accuracy: 0.7986170756731646


Output

```
Regex tagger, with default tagger as backoff
Brown 90% | NPS 10%
Accuracy: 0.0
Brown 50% | NPS 50%
Accuracy: 0.0
NPS 90% | BROWN 10%
Accuracy: 0.0
NPS 50% | BROWN 50%
Accuracy: 0.0
N-gram taggers with regex taggers as backoff
Brown 90% | NPS 10%
Accuracy: 0.7063558431155419
Brown 50% | NPS 50%
Accuracy: 0.8230555151097992
NPS 90% | BROWN 10%
Accuracy: 0.799443982872116
NPS 50% | BROWN 50%
Accuracy: 0.7986170756731646
````


#### c)

In [136]:
datasplit = datasetsTest["Brown 90% | NPS 10%"]
regTagger = regexTaggerWithBackoff(defaultTagger(getMostCommonTag(datasplit)))

ngramTagger = unigramTaggerWithBigramBackoffWithRegexBackoff(datasplit)

print("Regex with default backoff tagger")
print(regTagger.evaluate_per_tag(datasplit, truncate=5, sort_by_count=True))

print("Ngram tagger")
print(ngramTagger.evaluate_per_tag(datasplit, truncate=5, sort_by_count=True))

Regex with default backoff tagger
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.0000 | 0.0000 | 0.0000
VERB | 0.0000 | 0.0000 | 0.0000
   . | 0.0000 | 0.0000 | 0.0000
 ADP | 0.0000 | 0.0000 | 0.0000
 DET | 0.0000 | 0.0000 | 0.0000

Ngram tagger
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9709 | 0.9509 | 0.9608
VERB | 0.9751 | 0.9509 | 0.9628
   . | 0.9991 | 0.9891 | 0.9941
 ADP | 0.9364 | 0.9064 | 0.9212
 DET | 0.9865 | 0.9836 | 0.9850



Output

```
Regex with default backoff tagger
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.0000 | 0.0000 | 0.0000
VERB | 0.0000 | 0.0000 | 0.0000
   . | 0.0000 | 0.0000 | 0.0000
 ADP | 0.0000 | 0.0000 | 0.0000
 DET | 0.0000 | 0.0000 | 0.0000

Ngram tagger
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9709 | 0.9509 | 0.9608
VERB | 0.9751 | 0.9509 | 0.9628
   . | 0.9991 | 0.9891 | 0.9941
 ADP | 0.9364 | 0.9064 | 0.9212
 DET | 0.9865 | 0.9836 | 0.9850
```


#### d)

In [137]:
brownWordsTagged = getBrownWordsWithTag()

fd = nltk.FreqDist(brown.words())
cfd = nltk.ConditionalFreqDist(brownWordsTagged)

most_freq_words = fd.most_common(200)
likely_tags = dict((word, cfd[word].max()) for (word, _) in most_freq_words)

baselineTagger = UnigramTagger(model=likely_tags, backoff=defaultTagger(getMostCommonTag(getBrownTaggedSentences())))

for dKey in datasetsTest.keys():
    print(dKey)
    print("Accuracy: " + str(baselineTagger.accuracy(datasetsTest[dKey])))


Brown 90% | NPS 10%
Accuracy: 0.7702244128534367
Brown 50% | NPS 50%
Accuracy: 0.7623433094629833
NPS 90% | BROWN 10%
Accuracy: 0.7076628107624465
NPS 50% | BROWN 50%
Accuracy: 0.7623829878250524


Output

```
Brown 90% | NPS 10%
Accuracy: 0.7702244128534367
Brown 50% | NPS 50%
Accuracy: 0.7623433094629833
NPS 90% | BROWN 10%
Accuracy: 0.7076628107624465
NPS 50% | BROWN 50%
Accuracy: 0.7623829878250524
```


#### e)

In [138]:
from nltk.corpus import twitter_samples
from nltk.tokenize import word_tokenize

twitterSents = twitter_samples.strings()

for x in range(10):
   tokens = word_tokenize(twitterSents[x])
   print(baselineTagger.tag(tokens))

[('hopeless', 'NOUN'), ('for', 'ADP'), ('tmr', 'NOUN'), (':', '.'), ('(', '.')]
[('Everything', 'NOUN'), ('in', 'ADP'), ('the', 'DET'), ('kids', 'NOUN'), ('section', 'NOUN'), ('of', 'ADP'), ('IKEA', 'NOUN'), ('is', 'VERB'), ('so', 'ADV'), ('cute', 'NOUN'), ('.', '.'), ('Shame', 'NOUN'), ('I', 'PRON'), ("'m", 'NOUN'), ('nearly', 'NOUN'), ('19', 'NOUN'), ('in', 'ADP'), ('2', 'NOUN'), ('months', 'NOUN'), (':', '.'), ('(', '.')]
[('@', 'NOUN'), ('Hegelbon', 'NOUN'), ('That', 'NOUN'), ('heart', 'NOUN'), ('sliding', 'NOUN'), ('into', 'ADP'), ('the', 'DET'), ('waste', 'NOUN'), ('basket', 'NOUN'), ('.', '.'), (':', '.'), ('(', '.')]
[('“', 'NOUN'), ('@', 'NOUN'), ('ketchBurning', 'NOUN'), (':', '.'), ('I', 'PRON'), ('hate', 'NOUN'), ('Japanese', 'NOUN'), ('call', 'NOUN'), ('him', 'PRON'), ('``', '.'), ('bani', 'NOUN'), ("''", '.'), (':', '.'), ('(', '.'), (':', '.'), ('(', '.'), ('”', 'NOUN'), ('Me', 'NOUN'), ('too', 'ADV')]
[('Dang', 'NOUN'), ('starting', 'NOUN'), ('next', 'NOUN'), ('week', '

Output

````
[('hopeless', 'NOUN'), ('for', 'ADP'), ('tmr', 'NOUN'), (':', '.'), ('(', '.')]
[('Everything', 'NOUN'), ('in', 'ADP'), ('the', 'DET'), ('kids', 'NOUN'), ('section', 'NOUN'), ('of', 'ADP'), ('IKEA', 'NOUN'), ('is', 'VERB'), ('so', 'ADV'), ('cute', 'NOUN'), ('.', '.'), ('Shame', 'NOUN'), ('I', 'PRON'), ("'m", 'NOUN'), ('nearly', 'NOUN'), ('19', 'NOUN'), ('in', 'ADP'), ('2', 'NOUN'), ('months', 'NOUN'), (':', '.'), ('(', '.')]
[('@', 'NOUN'), ('Hegelbon', 'NOUN'), ('That', 'NOUN'), ('heart', 'NOUN'), ('sliding', 'NOUN'), ('into', 'ADP'), ('the', 'DET'), ('waste', 'NOUN'), ('basket', 'NOUN'), ('.', '.'), (':', '.'), ('(', '.')]
[('“', 'NOUN'), ('@', 'NOUN'), ('ketchBurning', 'NOUN'), (':', '.'), ('I', 'PRON'), ('hate', 'NOUN'), ('Japanese', 'NOUN'), ('call', 'NOUN'), ('him', 'PRON'), ('``', '.'), ('bani', 'NOUN'), ("''", '.'), (':', '.'), ('(', '.'), (':', '.'), ('(', '.'), ('”', 'NOUN'), ('Me', 'NOUN'), ('too', 'ADV')]
[('Dang', 'NOUN'), ('starting', 'NOUN'), ('next', 'NOUN'), ('week', 'NOUN'), ('I', 'PRON'), ('have', 'VERB'), ('``', '.'), ('work', 'NOUN'), ("''", '.'), (':', '.'), ('(', '.')]
[('oh', 'NOUN'), ('god', 'NOUN'), (',', '.'), ('my', 'DET'), ('babies', 'NOUN'), ("'", 'NOUN'), ('faces', 'NOUN'), (':', '.'), ('(', '.'), ('https', 'NOUN'), (':', '.'), ('//t.co/9fcwGvaki0', 'NOUN')]
[('@', 'NOUN'), ('RileyMcDonough', 'NOUN'), ('make', 'VERB'), ('me', 'PRON'), ('smile', 'NOUN'), (':', '.'), ('(', '.'), ('(', '.')]
[('@', 'NOUN'), ('f0ggstar', 'NOUN'), ('@', 'NOUN'), ('stuartthull', 'NOUN'), ('work', 'NOUN'), ('neighbour', 'NOUN'), ('on', 'ADP'), ('motors', 'NOUN'), ('.', '.'), ('Asked', 'NOUN'), ('why', 'NOUN'), ('and', 'CONJ'), ('he', 'PRON'), ('said', 'VERB'), ('hates', 'NOUN'), ('the', 'DET'), ('updates', 'NOUN'), ('on', 'ADP'), ('search', 'NOUN'), (':', '.'), ('(', '.'), ('http', 'NOUN'), (':', '.'), ('//t.co/XvmTUikWln', 'NOUN')]
[('why', 'NOUN'), ('?', '.'), (':', '.'), ('(', '.'), ('``', '.'), ('@', 'NOUN'), ('tahuodyy', 'NOUN'), (':', '.'), ('sialan', 'NOUN'), (':', '.'), ('(', '.'), ('https', 'NOUN'), (':', '.'), ('//t.co/Hv1i0xcrL2', 'NOUN'), ("''", '.')]
[('Athabasca', 'NOUN'), ('glacier', 'NOUN'), ('was', 'VERB'), ('there', 'PRT'), ('in', 'ADP'), ('#', 'NOUN'), ('1948', 'NOUN'), (':', '.'), ('-', 'NOUN'), ('(', '.'), ('#', 'NOUN'), ('athabasca', 'NOUN'), ('#', 'NOUN'), ('glacier', 'NOUN'), ('#', 'NOUN'), ('jasper', 'NOUN'), ('#', 'NOUN'), ('jaspernationalpark', 'NOUN'), ('#', 'NOUN'), ('alberta', 'NOUN'), ('#', 'NOUN'), ('explorealberta', 'NOUN'), ('#', 'NOUN'), ('…', 'NOUN'), ('http', 'NOUN'), (':', '.'), ('//t.co/dZZdqmf7Cz', 'NOUN')]
```


#### f)

In [139]:
brownTrain20, brownTest20 = split(getBrownTaggedSentences(), train_size=0.2)
brownTrain60, brownTest60 = split(getBrownTaggedSentences(), train_size=0.6)

taggerSmallTraining = unigramTaggerWithBigramBackoffWithRegexBackoff(brownTrain20)
taggerBiggerTraining = unigramTaggerWithBigramBackoffWithRegexBackoff(brownTrain60)

print("Small training data set")
print(taggerSmallTraining.evaluate_per_tag(getBrownTaggedSentences(), truncate=5, sort_by_count=5))

print("Bigger training data set")
print(taggerBiggerTraining.evaluate_per_tag(getBrownTaggedSentences(), truncate=5, sort_by_count=5))

Small training data set
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9623 | 0.5645 | 0.7116
VERB | 0.9751 | 0.6531 | 0.7823
   . | 0.9997 | 0.6407 | 0.7809
 ADP | 0.9253 | 0.7469 | 0.8266
 DET | 0.9852 | 0.8460 | 0.9103

Bigger training data set
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9699 | 0.8396 | 0.9001
VERB | 0.9741 | 0.8766 | 0.9228
   . | 0.9997 | 0.8852 | 0.9389
 ADP | 0.9365 | 0.8963 | 0.9160
 DET | 0.9870 | 0.9581 | 0.9723



Output

```
Small training data set
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9623 | 0.5645 | 0.7116
VERB | 0.9751 | 0.6531 | 0.7823
   . | 0.9997 | 0.6407 | 0.7809
 ADP | 0.9253 | 0.7469 | 0.8266
 DET | 0.9852 | 0.8460 | 0.9103

Bigger training data set
 Tag | Prec.  | Recall | F-measure
-----+--------+--------+-----------
NOUN | 0.9699 | 0.8396 | 0.9001
VERB | 0.9741 | 0.8766 | 0.9228
   . | 0.9997 | 0.8852 | 0.9389
 ADP | 0.9365 | 0.8963 | 0.9160
 DET | 0.9870 | 0.9581 | 0.9723
```


Even though it is a little bit obvious, it is fun to see more is better in NLTK. The bigger training data set you have, the more accurate your predictions will be.

### Ex 3

In [141]:
import nltk
from nltk.corpus import brown
# see https://nltk.readthedocs.io/en/latest/api/nltk.html
# define distinguishable start/end tuples of tag/word
# used to mark sentences
START = ("START", "START")
END = ("END", "END")

def get_tags(corpus):
    tags_words = []
    for sent in corpus.tagged_sents():
        tags_words.append(START)
        # shorten tags to 2 characters each for simplicity
        tags_words.extend([(tag[:2], word) for (word, tag) in sent])
        tags_words.append(END)

    return tags_words

def probDist(corpus, probability_distribution, tag_observation_fn):
    tag_words = get_tags(corpus)
    tags = [tag for (tag, _) in tag_words]
    # conditional frequency distribution over tag/word
    cfd_tagwords = nltk.ConditionalFreqDist(tag_words)
    cpd_tagwords = nltk.ConditionalProbDist(cfd_tagwords, probability_distribution)
    # conditional frequency distribution of observations:
    cfd_tags = nltk.ConditionalFreqDist(tag_observation_fn(tags))
    cpd_tags = nltk.ConditionalProbDist(cfd_tags, probability_distribution)
    
    return cpd_tagwords, cpd_tags

def task3a():
    corpus = brown
    # maximum likelihood estimate to create a probability distribution 
    probability_distribution = nltk.MLEProbDist
    # a function to create tag observations. Hint: can we observe anything with unigrams?
    tag_observation_fn = nltk.bigrams
    return probDist(corpus, probability_distribution, tag_observation_fn)
    
def prettify(prob):
    return "{}%".format(round(prob * 100, 4))
    

#### b)

In [154]:
tagwords, tags = task3a()

def task3b():
    prob_verb_is_run = tagwords["VB"].prob("run")  
    prob_v_follows_p = tags["VB"].prob("PP")
    print("Prob. of a Verb(VB) being 'run' is", prettify(prob_verb_is_run))
    print("Prob. of a Preposition(PP) being followed by a Verb(VB) is", prettify(prob_v_follows_p))

task3b()

Prob. of a Verb(VB) being 'run' is 0.1329%
Prob. of a Preposition(PP) being followed by a Verb(VB) is 10.9951%


Output

````
Prob. of a Verb(VB) being 'run' is 0.1329%
Prob. of a Preposition(PP) being followed by a Verb(VB) is 10.9951%
````


#### c)

In [160]:
print("Ten most common words for NN, VB, and JJ")
t = ["NN", "VB", "JJ"]
for tag in t:
    print(tagwords[tag].freqdist().most_common(10))

Ten most common words for NN, VB, and JJ
[('time', 1555), ('man', 1148), ('Af', 994), ('years', 942), ('way', 883), ('people', 809), ('men', 736), ('world', 684), ('life', 676), ('year', 647)]
[('said', 1943), ('made', 1119), ('make', 765), ('see', 727), ('get', 719), ('know', 676), ('came', 621), ('used', 610), ('go', 604), ('come', 589)]
[('new', 1060), ('such', 903), ('own', 750), ('good', 693), ('great', 592), ('New', 575), ('old', 568), ('American', 535), ('small', 517), ('long', 515)]


Output

````
Ten most common words for NN, VB, and JJ
[('time', 1555), ('man', 1148), ('Af', 994), ('years', 942), ('way', 883), ('people', 809), ('men', 736), ('world', 684), ('life', 676), ('year', 647)]
[('said', 1943), ('made', 1119), ('make', 765), ('see', 727), ('get', 719), ('know', 676), ('came', 621), ('used', 610), ('go', 604), ('come', 589)]
[('new', 1060), ('such', 903), ('own', 750), ('good', 693), ('great', 592), ('New', 575), ('old', 568), ('American', 535), ('small', 517), ('long', 515)]
````


#### d)

In [173]:
# PP VB VB DT NN - I can code some code

prob = tags["START"].prob("PP")

prob *= tags["PP"].prob("VB") * tagwords["PP"].prob("I")
prob *= tags["VB"].prob("VB") * tagwords["VB"].prob("can")
prob *= tags["VB"].prob("DT") * tagwords["VB"].prob("code")
prob *= tags["DT"].prob("NN") * tagwords["DT"].prob("some")
prob *= tags["NN"].prob("END") * tagwords["NN"].prob("code")

print(prob)

2.1809718128948064e-24


Output

```
2.1809718128948064e-24
```

Very low it seems
